# 04 · Model Serving y consumo REST API — Adult Income

**Objetivo:** desplegar la versión `Champion` en un endpoint serverless y consumir su API REST (desde el propio notebook, desde Postman y luego desde la web local).

> Ejecuta primero los notebooks 01, 02 y 03. La creación inicial del endpoint puede tardar varios minutos. Free Edition limita la cantidad/capacidad de endpoints y no ofrece GPU; un modelo sklearn pequeño sí encaja en esta práctica.


## 1. Configuración y versión a desplegar

Model Serving despliega una versión concreta. Resolvemos el alias `Champion` al inicio para saber exactamente qué versión quedará activa.


In [0]:
import mlflow
from mlflow import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

# --- Configuración -----------------------------------------------------------
catalog = spark.catalog.currentCatalog()          # detecta el catálogo activo
schema = "mlops_income_course"
model_name = "income_classifier"
endpoint_name = "mlops-course-income"
model_alias = "Champion"

registered_model_name = f"{catalog}.{schema}.{model_name}"

# --- Resolver la versión Champion -------------------------------------------
mlflow.set_registry_uri("databricks-uc")
registry_client = MlflowClient(registry_uri="databricks-uc")

champion_version = registry_client.get_model_version_by_alias(
    name=registered_model_name,
    alias=model_alias,
)
model_version = champion_version.version

# --- Mostrar resultado -------------------------------------------------------
print(f"Endpoint        : {endpoint_name}")
print(f"Modelo          : {registered_model_name}")
print(f"Versión Champion: {model_version}")


## 2. Crear o actualizar el endpoint

`Small` y scale-to-zero reducen el consumo para la demo. La celda puede tardar mientras Databricks construye el entorno del modelo. Si el endpoint ya existe, lo actualiza a la versión Champion actual (idempotente).


In [0]:
from databricks.sdk.errors import NotFound

w = WorkspaceClient()

# --- Entidad servida --------------------------------------------------------
served_entity = ServedEntityInput(
    name="income-champion",
    entity_name=registered_model_name,
    entity_version=str(model_version),
    workload_size="Small",
    scale_to_zero_enabled=True,
)

endpoint_config = EndpointCoreConfigInput(served_entities=[served_entity])

# --- Crear o actualizar (idempotente) --------------------------------------
try:
    w.serving_endpoints.get(endpoint_name)
    exists = True
except NotFound:
    exists = False

if exists:
    print(f"Actualizando endpoint '{endpoint_name}' a la versión {model_version}…")
    w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_entities=[served_entity],
    )
else:
    print(f"Creando endpoint '{endpoint_name}' con la versión {model_version}…")
    w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=endpoint_config,
    )

# --- Comprobar estado -------------------------------------------------------
endpoint = w.serving_endpoints.get(endpoint_name)
print(f"Endpoint : {endpoint_name}")
print(f"¿Listo?  : {endpoint.state.ready}")


## 3. Payload de inferencia

El contrato usa el formato `dataframe_records`: una lista de registros con exactamente las siete columnas de la firma MLflow (las 6 features base + `capital_net_ratio`). No enviamos `client_id` porque esta clase no configura lookups online.

Dos casos de prueba inventados, pensados para dar resultados contrastantes en la demo.


In [0]:
import json

# --- Payload de inferencia (dataframe_records) con dos casos de prueba -----
records = [
    {
        # Candidato "califica" (debería predecir 1)
        "age": 45,
        "education_num": 13,
        "hours_per_week": 55,
        "capital_gain": 15000,
        "capital_loss": 0,
        "fnlwgt": 190000,
        "capital_net_ratio": (15000 - 0) / (55 + 1),
    },
    {
        # Candidato "no califica" (debería predecir 0)
        "age": 22,
        "education_num": 9,
        "hours_per_week": 20,
        "capital_gain": 0,
        "capital_loss": 0,
        "fnlwgt": 210000,
        "capital_net_ratio": (0 - 0) / (20 + 1),
    },
]

payload = {"dataframe_records": records}

# --- Mostrar el payload -----------------------------------------------------
print(json.dumps(payload, indent=2))


## 4. Consumir la REST API (desde el propio notebook)

In [0]:
import json

# --- Invocar la API REST con el cliente autenticado ------------------------
# El SDK reutiliza la identidad del notebook: no se exponen tokens ni credenciales.
response = w.serving_endpoints.query(
    name=endpoint_name,
    dataframe_records=payload["dataframe_records"],
)

# --- Mostrar la respuesta --------------------------------------------------
print(f"Predicciones: {response.predictions}")
print(f"  Caso 1 (califica esperado 1)    → {response.predictions[0]}")
print(f"  Caso 2 (no califica esperado 0) → {response.predictions[1]}")


## 5. Probar en Postman

Fuera del notebook, el contrato HTTP es el mismo:

- **Method:** `POST`
- **URL:** `https://<tu-host-de-databricks>/serving-endpoints/mlops-course-income/invocations`
  (el host aparece en la barra de direcciones de tu workspace, ej. `dbc-xxxxxxxx-xxxx.cloud.databricks.com`)
- **Headers:**
  - `Authorization: Bearer <tu-personal-access-token>`
  - `Content-Type: application/json`
- **Body (raw JSON):** copia el JSON impreso en la celda de la sección 3 (el diccionario `payload` completo, con `dataframe_records`).

Genera el token en **Settings → Developer → Access tokens** de tu workspace. Nunca lo pongas directamente en código versionado en GitHub — en el backend local usaremos variables de entorno (`DATABRICKS_HOST`, `DATABRICKS_TOKEN`).


## Cierre de la clase

```text
Adult Income (OpenML)
  → Delta + Feature Store
  → training set
  → MLflow Experiments (5 modelos comparados)
  → mejor run
  → Unity Catalog Model Registry + Champion
  → Model Serving
  → POST REST (notebook, Postman, y luego backend local + HTML)
```

**Siguiente paso:** backend local (Flask/FastAPI) que haga de proxy hacia este endpoint, más un formulario HTML simple para la demo visual.
